# Notebook 06 — Observability

Shows how per-request traces make workflow behavior inspectable.

<!-- TODO main-session: expand intro -->

---

## Setup

Loads the repo root, environment, and public observability APIs used throughout this notebook.

<!-- TODO main-session: expand teaching framing -->

---

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.workflow import run_workflow
from src.llm import LLMClient
from src.rag import ingest
from src.observability import (
    Span,
    Trace,
    SessionMetrics,
    LocalTraceStore,
    get_store,
    trace_workflow,
    compute_metrics,
)

has_key = bool(os.getenv("ANTHROPIC_API_KEY"))
print(f"Anthropic key present: {has_key}")

Anthropic key present: False


## Corpus

Indexes the sample program corpus into a notebook-local vector store for the traced workflow call.

<!-- TODO main-session: expand teaching framing -->

---

In [2]:
persist_dir = repo_root / "data" / "chroma_nb06"
result = ingest(repo_root / "data", persist_dir)
print(result)

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


IngestionResult(documents_loaded=5, chunks_created=42, chunks_indexed=42, vector_store_path=WindowsPath('C:/Users/narla/OneDrive/Desktop/TalentSprint/IISc_GenAI_C2/LLMOps/llmops-session/data/chroma_nb06'), embedding_model='sentence-transformers/all-MiniLM-L6-v2')


## One traced workflow call

Wraps a single workflow run so the trace and its top-level observability fields are visible together.

<!-- TODO main-session: expand teaching framing -->

---

In [3]:
from IPython.display import display
import pandas as pd

llm = LLMClient() if has_key else LLMClient(provider="mock")
store = LocalTraceStore()
trace = trace_workflow(
    run_workflow,
    question="What is the late submission policy?",
    persist_dir=persist_dir,
    llm=llm,
    store=store,
)

if not has_key:
    print("Using mock client; token counts may be zero.")

display(trace)
with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
    display(store.to_dataframe())

C:\Users\narla\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
Unexpected classification returned by LLM; falling back to out_of_scope. raw_output='[mock:59bb19f5] echo: You are a question classifier for the TalentSpri...'


Workflow trace entries did not expose cache_status; leaving Trace.cache_status empty.


Using mock client; token counts may be zero.


Trace(trace_id='9fa854f7', query='What is the late submission policy?', start_ms=320207509.0289, spans=[Span(name='classify', start_ms=320207509.0289, end_ms=320207509.1363, attributes={'input': 'What is the late submission policy?', 'output': 'out_of_scope', 'latency_ms': 0.10740000288933516, 'prompt_version': 'v1', 'route_to': 'refuse'}), Span(name='refuse', start_ms=320207509.1363, end_ms=320207512.1362001, attributes={'input': 'What is the late submission policy?', 'output': 'Thanks for asking, but that question is outside the scope of this program assistant.\nPlease use the appropriate professional or official resource for:\n"What is the late submission policy?"\n', 'classification': 'out_of_scope', 'latency_ms': 2.9999000253155828, 'prompt_version': 'v1'})], category='out_of_scope', backend='mock', model='mock-model-v1', prompt_tokens=0, completion_tokens=0, total_tokens=0, latency_ms=249.2, retrieved_count=0, cache_status='', refused=True, escalated=False, guardrail_input_flag='

,trace_id,query,category,backend,model,prompt_tokens,completion_tokens,total_tokens,latency_ms,retrieved_count,cache_status,refused,escalated,guardrail_input_flag,guardrail_output_flag,workflow_steps
0,9fa854f7,What is the late submission policy?,out_of_scope,mock,mock-model-v1,0,0,0,249.2,0,,True,False,,,classify → refuse


## Spans — the per-node narrative inside the trace

Prints the ordered spans captured for the traced workflow call above.

<!-- TODO main-session: expand -->

In [4]:
for i, span in enumerate(trace.spans):
    print(
        f"[{i}] {span.name:15s} duration_ms={span.duration_ms:7.2f}  "
        f"attributes={dict(span.attributes)}"
    )
print(f"\nworkflow_steps = {trace.workflow_steps}")

[0] classify        duration_ms=   0.11  attributes={'input': 'What is the late submission policy?', 'output': 'out_of_scope', 'latency_ms': 0.10740000288933516, 'prompt_version': 'v1', 'route_to': 'refuse'}
[1] refuse          duration_ms=   3.00  attributes={'input': 'What is the late submission policy?', 'output': 'Thanks for asking, but that question is outside the scope of this program assistant.\nPlease use the appropriate professional or official resource for:\n"What is the late submission policy?"\n', 'classification': 'out_of_scope', 'latency_ms': 2.9999000253155828, 'prompt_version': 'v1'}

workflow_steps = ['classify', 'refuse']


## What each span attribute means

PLAN.md S-12 defines the per-node attributes that make the workflow debuggable from a saved trace.

<!-- TODO main-session: expand -->